# AUDIT — Stage 01 QC (+00b inputs, 01c concat)
**Read-only.** Sources of truth: `workflow/scripts/ingest_qc/qc.py`, `config/config.yaml` `qc:` block,
Cevrim et al. Methods. This notebook (1) echoes the recipe, (2) verifies every surviving cell
satisfies the hard thresholds, (3) **replays the sequential filter arithmetic from the raw 00b
objects** and compares survivor counts, (4) reconciles the cell-count ladder from post-QC through
the atlas objects to the paper's `published_n`. A FAIL means code and data disagree — investigate
before trusting anything downstream.

In [ ]:
import json, glob, os, re, pathlib
import numpy as np, pandas as pd, h5py, yaml
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from anndata.io import read_elem
X = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
CHECKS = []
def check(label, ok, got=None, want=None, warn=False):
    CHECKS.append(dict(label=label, ok=bool(ok), warn=warn, got=str(got)[:160], want=str(want)[:120]))
    print(("PASS " if ok else ("WARN " if warn else "FAIL ")) + f"{label}: got={got}" + (f" want={want}" if want is not None else ""))
def summary(tag):
    bad = [c for c in CHECKS if not c["ok"] and not c["warn"]]
    warns = [c for c in CHECKS if not c["ok"] and c["warn"]]
    json.dump(CHECKS, open(f"audit_{tag}_report.json", "w"), indent=1)
    print(f"\n===== {len(CHECKS)} checks | {len(bad)} FAIL | {len(warns)} WARN =====")
    for c in bad + warns: print(("FAIL " if not c["warn"] else "WARN ") + c["label"])
cfg = yaml.safe_load(open(X/"config/config.yaml"))

In [ ]:
# --- recipe echo: config vs the constants qc.py documents ---
qc = cfg["qc"]; print(json.dumps(qc, indent=1))
check("config qc.min_counts", qc.get("min_counts") == 500, qc.get("min_counts"), 500)
check("config qc.n_mads", qc.get("n_mads") == 3, qc.get("n_mads"), 3)
check("config qc.max_pct_mito", qc.get("max_pct_mito") == 8, qc.get("max_pct_mito"), 8)
src = open(X/"workflow/scripts/ingest_qc/qc.py").read()
for token, why in [("median_abs_deviation", "raw MAD (no 1.4826)"), ("log10", "log10(x+1) scale"),
                   ("min_counts", "hard floor first"), ("BoostClassifier", "DoubletDetection")]:
    check(f"qc.py contains {token} ({why})", token in src, token in src, True)

In [ ]:
# --- discover samples; pair raw (00b) with qc'd (01) ---
qcd = sorted(glob.glob(str(X/"data/interim/qc/*.h5ad")))
rows = []
for p in qcd:
    s = os.path.basename(p)[:-5]
    raw = X/f"data/interim/raw/{s}.h5ad"
    rows.append(dict(sample=s, qc_gb=round(os.path.getsize(p)/1e9,2), raw_exists=raw.exists()))
S = pd.DataFrame(rows).set_index("sample")
print(S.to_string())
check("every QC'd sample has its raw 00b input", S.raw_exists.all(), f"{int(S.raw_exists.sum())}/{len(S)}")

In [ ]:
# --- survivor checks: every retained cell satisfies the hard thresholds ---
surv = {}
for s in S.index:
    with h5py.File(X/f"data/interim/qc/{s}.h5ad") as f:
        obs = {c: read_elem(f[f"obs/{c}"]) for c in ("total_counts", "pct_counts_mt") if c in f["obs"]}
        dd = read_elem(f["obs/doubletdetection_predicted_doublet"]) if "doubletdetection_predicted_doublet" in f["obs"] else None
        n = f["obs/_index"].shape[0]
    surv[s] = dict(n=n, min_counts=float(np.min(obs["total_counts"])), max_mito=float(np.max(obs["pct_counts_mt"])),
                   doublets_retained=(int(pd.Series(dd).astype(str).isin(["True","1","1.0"]).sum()) if dd is not None else None))
V = pd.DataFrame(surv).T
print(V.to_string())
check("all survivors >= min_counts floor", (V.min_counts >= qc["min_counts"]).all(), V.min_counts.min(), f">= {qc['min_counts']}")
check("all survivors <= max mito%", (V.max_mito <= qc["max_pct_mito"] + 1e-9).all(), round(V.max_mito.max(),3), f"<= {qc['max_pct_mito']}")
if V.doublets_retained.notna().any():
    check("no predicted doublet retained", (V.doublets_retained.fillna(0) == 0).all(), int(V.doublets_retained.fillna(0).sum()), 0)

In [ ]:
# --- replay the sequential filter from raw, per sample (exact re-derivation) ---
import scanpy as sc
from scipy.stats import median_abs_deviation
def replay(s):
    a = sc.read_h5ad(X/f"data/interim/raw/{s}.h5ad")
    a.var["mt"] = a.var_names.str.startswith(("mt-", "MT-"))
    a.var["ribo"] = a.var_names.str.startswith(("rps","rpl","mrps","mrpl","Rps","Rpl","Mrps","Mrpl","RPS","RPL","MRPS","MRPL"))
    a.var["hb"] = a.var_names.str.contains("^Hb[^(p)]") | a.var_names.str.contains("^HB[^(P)]")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt","ribo","hb"], percent_top=[20], log1p=True, inplace=True)
    n0 = a.n_obs
    a = a[a.obs.total_counts >= qc["min_counts"]].copy()          # 1. floor (mutates, like qc.py)
    n_floor = a.n_obs
    a = a[a.obs.pct_counts_mt <= qc["max_pct_mito"]].copy()      # 2. mito
    n_mito = a.n_obs
    for col in ("total_counts", "n_genes_by_counts"):             # 3+4. sequential raw-MAD on log10(x+1)
        v = np.log10(a.obs[col].to_numpy() + 1)
        med, mad = np.nanmedian(v), median_abs_deviation(v, nan_policy="omit")
        a = a[(v >= med - qc["n_mads"]*mad) & (v <= med + qc["n_mads"]*mad)].copy()
    return dict(n_raw=n0, n_floor=n_floor, n_mito=n_mito, n_filters=a.n_obs)
rep = {}
for s in S.index[S.raw_exists]:
    rep[s] = replay(s)
    rep[s]["n_qcd"] = surv[s]["n"]
    rep[s]["doublets_implied"] = rep[s]["n_filters"] - rep[s]["n_qcd"]
    print(s, rep[s], flush=True)
R = pd.DataFrame(rep).T
# qc'd survivors = replayed filter survivors minus doublets; doublets_implied must be >= 0 and modest
check("replayed filters >= stored survivors (doublets non-negative)", (R.doublets_implied >= 0).all(),
      f"min implied doublets {int(R.doublets_implied.min())}")
check("implied doublet fraction < 15% everywhere", (R.doublets_implied / R.n_filters < 0.15).all(),
      round(float((R.doublets_implied / R.n_filters).max()), 4), warn=True)
# cross-check implied doublet counts against the stage logs where they exist
for s in R.index:
    lg = X/f"logs/01_qc_{s}.log"
    if lg.exists():
        m = re.findall(r"(\d+)\s+doublets", open(lg).read())
        if m: check(f"{s}: log doublets == implied", int(m[-1]) == int(R.loc[s,"doublets_implied"]),
                    f"log {m[-1]} vs implied {int(R.loc[s,'doublets_implied'])}")

In [ ]:
# --- the count ladder: post-QC -> concat -> atlas -> published_n ---
ladder = {}
for sp, pat in (("mouse", ("uninduced","_dpi")), ("human", ("menstrual","secretory"))):
    mine = [s for s in S.index if any(t in s for t in pat)]
    ladder[sp] = {"post_qc_sum": int(sum(surv[s]["n"] for s in mine)), "n_samples": len(mine)}
    for tag, p in (("concat", X/f"data/interim/qc_passed_{sp}.h5ad"),
                   ("latent(02)", X/f"data/interim/latent_{sp}.h5ad"),
                   ("final", X/f"data/interim/final_{sp}.h5ad")):
        if p.exists():
            with h5py.File(p) as f: ladder[sp][tag] = int(f["obs/_index"].shape[0])
    ladder[sp]["published_n"] = cfg.get("published_n", {}).get(sp)
L = pd.DataFrame(ladder).T
print(L.to_string())
for sp in ladder:
    if "concat" in ladder[sp]:
        check(f"{sp}: concat == sum of post-QC samples", ladder[sp]["concat"] == ladder[sp]["post_qc_sum"],
              f"{ladder[sp]['concat']} vs {ladder[sp]['post_qc_sum']}")
    if "latent(02)" in ladder[sp] and "concat" in ladder[sp]:
        check(f"{sp}: integrate loses no cells", ladder[sp]["latent(02)"] == ladder[sp]["concat"],
              f"{ladder[sp]['latent(02)']} vs {ladder[sp]['concat']}")
    if ladder[sp].get("published_n") and "latent(02)" in ladder[sp]:
        d = ladder[sp]["latent(02)"] - ladder[sp]["published_n"]
        check(f"{sp}: current vs published_n (delta {d:+d})", d == 0, ladder[sp]["latent(02)"],
              ladder[sp]["published_n"], warn=True)

In [ ]:
# --- figures: per-sample QC distributions with thresholds drawn ---
ns = len(S.index); fig, axes = plt.subplots(2, ns, figsize=(1.6*ns, 5), sharey="row")
for j, s in enumerate(S.index):
    with h5py.File(X/f"data/interim/qc/{s}.h5ad") as f:
        tc = read_elem(f["obs/total_counts"]); mt = read_elem(f["obs/pct_counts_mt"])
    axes[0, j].hist(np.log10(tc + 1), bins=50); axes[0, j].axvline(np.log10(qc["min_counts"]+1), color="r", lw=.8)
    axes[1, j].hist(mt, bins=50); axes[1, j].axvline(qc["max_pct_mito"], color="r", lw=.8)
    axes[0, j].set_title(s, fontsize=6, rotation=45, ha="left"); axes[0, j].set_xticks([]); axes[1, j].set_xlabel("")
axes[0, 0].set_ylabel("log10 counts"); axes[1, 0].set_ylabel("pct mito")
fig.tight_layout(); fig.savefig("audit_01_qc_distributions.png", dpi=150, bbox_inches="tight")
print("wrote audit_01_qc_distributions.png")

In [ ]:
summary("01_qc")